# Day 05 · Regression beyond a line

**Goal:** handle many features at once, let the fit curve (polynomial features), detect multicollinearity with VIF, *see* under/overfitting and the bias-variance trade-off, and tame it with **Ridge** and **Lasso**.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`.

**Runtime:** CPU is fine (everything here trains in seconds). No GPU needed.

| Section | What you'll do |
|---|---|
| 1 | Multiple linear regression on California housing |
| 2 | Read the coefficients (and why they can lie) |
| 3 | Multicollinearity & VIF |
| 4 | Polynomial regression and the overfitting trap |
| 5 | Bias-variance, seen with your own eyes |
| 6 | Ridge vs Lasso: coefficient paths |
| 7 | Picking alpha with cross-validation |
| 8 | 🎯 Try it yourself |
| 9 | Homework starter + key takeaways |

## 0. Setup

Everything we need ships with Colab: `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `statsmodels`. We set a seed so your numbers match the instructor's.

In [ ]:
# All preinstalled on Colab; nothing to pip install today.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, statsmodels
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (8, 4.5)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__, "| statsmodels", statsmodels.__version__)

## 1. Multiple linear regression on California housing 🏠

**What:** 20,640 census districts in California, 8 numeric features (median income, house age, rooms per household, latitude/longitude ...), target = median house value in units of **$100,000**.

**Why this dataset:** it is built into scikit-learn (downloads once, ~1 MB), it is real, and it is *not* clean-room perfect: some features are correlated, some are useless, and the relationship is not a straight line. Exactly the problems today is about.

The model is the same `LinearRegression` as Day 04. The only difference is `X` now has 8 columns instead of 1.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
print(X.shape, "features:", list(X.columns))
X.describe().T.round(2)

In [ ]:
# Hold out 20% that the model never sees. This is how we will catch overfitting later.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

lr = LinearRegression().fit(X_tr, y_tr)
pred = lr.predict(X_te)

rmse = mean_squared_error(y_te, pred) ** 0.5
print(f"Test RMSE : {rmse:.3f}  (≈ ${rmse*100:.0f}k typical error)")
print(f"Test R²   : {r2_score(y_te, pred):.3f}")
print(f"Train R²  : {lr.score(X_tr, y_tr):.3f}   <- close to test R², so no overfitting here")

## 2. Reading the coefficients (carefully)

Each coefficient is "how much the prediction changes when this feature goes up by 1 unit, holding the others fixed". That last clause is the whole trick of multiple regression, and it is also where interpretation goes wrong: **if two features move together, "holding the other fixed" never happens in the data**, and the weights become unreliable.

Look at `AveRooms` and `AveBedrms` below: one positive, one negative, both large. Does a bedroom really *lower* the price? No. The two features are correlated and the model is splitting the credit in a strange way.

In [ ]:
coef = pd.Series(lr.coef_, index=X.columns).sort_values()
print(coef.round(3))
print("intercept:", round(lr.intercept_, 3))

coef.plot(kind="barh", color=["#FF6B8A" if v < 0 else "#4FD3F2" for v in coef])
plt.title("Raw coefficients (units differ per feature, so sizes are NOT comparable yet)")
plt.axvline(0, color="k", lw=.5); plt.show()

> 🧠 **Interview Q:** *"Can I compare coefficients to find the most important feature?"* — Only after **standardising** the features (mean 0, std 1). Otherwise a feature measured in thousands (Population) automatically gets a tiny weight and a feature measured in units (AveOccup) a big one. Let's redo it properly.

In [ ]:
# Standardise inside a pipeline so the scaler only ever sees training data.
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
std_coef = pd.Series(pipe[-1].coef_, index=X.columns).sort_values()
print(std_coef.round(3))
print("\nNow the magnitudes are comparable: MedInc dominates, Latitude/Longitude matter, Population barely does.")

## 3. Multicollinearity and VIF 🔍

**Variance Inflation Factor** for feature *j*: regress feature *j* on all the *other* features, take that R², and compute

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

If the others can predict feature *j* well, R² → 1 and VIF explodes. Rules of thumb: **VIF > 5** = warning, **VIF > 10** = fix it (drop one of the twins, combine them, or use Ridge).

We compute it twice: once by hand (so you own the formula) and once with `statsmodels` (the one-liner you will use at work).

In [ ]:
def vif_by_hand(df):
    out = {}
    for col in df.columns:
        others = df.drop(columns=col)
        r2 = LinearRegression().fit(others, df[col]).score(others, df[col])
        out[col] = 1 / (1 - r2 + 1e-12)          # epsilon guards against R² == 1
    return pd.Series(out).sort_values(ascending=False)

print(vif_by_hand(X_tr).round(2))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# statsmodels expects a constant column for the intercept, otherwise VIFs are inflated
Xc = X_tr.assign(const=1.0)
vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])], index=Xc.columns)
print(vif.drop("const").sort_values(ascending=False).round(2))
print("\nAveRooms / AveBedrms and Latitude / Longitude are the correlated pairs. Their individual weights are the ones not to trust.")

In [ ]:
# A quick visual confirmation: the correlation heatmap
import seaborn as sns
plt.figure(figsize=(7, 5.5))
sns.heatmap(X_tr.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature correlations (look for |r| > 0.8)"); plt.show()

## 4. Polynomial regression and the overfitting trap 🎢

**The trick:** add columns $x^2, x^3, \dots$ and run *ordinary* linear regression on them. The model is still linear in the weights; only the features are curved.

**The danger:** the degree is a complexity knob. Too low → the model can't follow the shape (**underfitting**, high bias). Too high → it follows the noise of *this* training set (**overfitting**, high variance).

We use a small synthetic 1-D dataset so you can *see* the curve. Real data is the same, just in more dimensions where you can't look.

In [ ]:
rng = np.random.default_rng(SEED)
true_f = lambda x: 0.8 * np.sin(3 * x) + 0.4 * x

x_all = np.sort(rng.uniform(-1, 1, 40))
y_all = true_f(x_all) + rng.normal(0, 0.25, 40)
x_tr1, x_te1, y_tr1, y_te1 = train_test_split(x_all[:, None], y_all, test_size=0.5, random_state=0)

grid = np.linspace(-1.05, 1.05, 300)[:, None]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
for ax, d in zip(axes, [1, 3, 6, 14]):
    model = make_pipeline(PolynomialFeatures(degree=d), StandardScaler(), LinearRegression()).fit(x_tr1, y_tr1)
    ax.scatter(x_tr1, y_tr1, c="#4FD3F2", label="train")
    ax.scatter(x_te1, y_te1, facecolors="none", edgecolors="#FFB43A", label="test")
    ax.plot(grid, true_f(grid), "k--", lw=1, label="truth")
    ax.plot(grid, np.clip(model.predict(grid), -3, 3), c="#FF6B8A", lw=2)
    tr = mean_squared_error(y_tr1, model.predict(x_tr1)); te = mean_squared_error(y_te1, model.predict(x_te1))
    ax.set_title(f"degree {d}\ntrain {tr:.3f} | test {te:.3f}"); ax.set_ylim(-2, 2)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# The degree sweep: this table IS the bias-variance trade-off
rows = []
for d in range(1, 16):
    model = make_pipeline(PolynomialFeatures(degree=d), StandardScaler(), LinearRegression()).fit(x_tr1, y_tr1)
    rows.append({"degree": d,
                 "train_mse": mean_squared_error(y_tr1, model.predict(x_tr1)),
                 "test_mse": mean_squared_error(y_te1, model.predict(x_te1)),
                 "max_|coef|": np.abs(model[-1].coef_).max()})
sweep = pd.DataFrame(rows).set_index("degree")
print(sweep.round(3))

sweep[["train_mse", "test_mse"]].clip(upper=3).plot(marker="o")
plt.ylabel("MSE (clipped at 3)"); plt.title("Train keeps falling, test U-turns: that gap is overfitting")
plt.show()
print("Best degree by TEST error:", sweep["test_mse"].idxmin())

## 5. Bias-variance, seen with your own eyes 👀

The decomposition says expected test error = **Bias²** + **Variance** + **noise**. You can't compute the three parts on real data, but you can *simulate* them: keep the true curve, redraw the noise 30 times, refit, and watch.

* A **degree-1** model gives almost the same wrong line every time → low variance, high bias.
* A **degree-14** model gives a wildly different curve every time → low bias on average, huge variance.

This is the mechanism behind the "resample noise" button on today's lesson page.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, d in zip(axes, [1, 4, 14]):
    fits = []
    for rep in range(30):
        r = np.random.default_rng(rep)
        xs = np.sort(r.uniform(-1, 1, 20)); ys = true_f(xs) + r.normal(0, 0.25, 20)
        m = make_pipeline(PolynomialFeatures(degree=d), StandardScaler(), LinearRegression()).fit(xs[:, None], ys)
        f = np.clip(m.predict(grid), -3, 3); fits.append(f)
        ax.plot(grid, f, c="#FF6B8A", alpha=.15, lw=1)
    fits = np.array(fits)
    ax.plot(grid, fits.mean(0), c="#FF6B8A", lw=2.5, label="average model")
    ax.plot(grid, true_f(grid), "k--", lw=1.5, label="truth")
    bias2 = np.mean((fits.mean(0) - true_f(grid).ravel()) ** 2)
    var = np.mean(fits.var(0))
    ax.set_title(f"degree {d}: bias² ≈ {bias2:.3f}, variance ≈ {var:.3f}"); ax.set_ylim(-2, 2)
axes[0].legend(); plt.tight_layout(); plt.show()
print("Degree 4 has the best SUM of bias² + variance. That is the whole point.")

## 6. Ridge vs Lasso: coefficient paths 🎚️

Overfitting has a fingerprint: **huge coefficients** (look at `max_|coef|` in the sweep table). Regularization adds a *tax* on coefficient size to the loss:

* **Ridge (L2):** $\;\text{loss} + \alpha \sum \theta_j^2$ → shrinks smoothly, never exactly zero. Also the classic fix for multicollinearity.
* **Lasso (L1):** $\;\text{loss} + \alpha \sum |\theta_j|$ → the corner at zero pushes weak weights to **exactly 0** = automatic feature selection.

**Always scale features first** (the penalty must treat every column fairly). Watch the coefficient *paths* below on California housing as alpha grows.

In [ ]:
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

alphas = np.logspace(-3, 1, 40)
ridge_paths = np.array([Ridge(alpha=a * len(Xs_tr) / 10).fit(Xs_tr, y_tr).coef_ for a in alphas])   # scaled so both plots span a similar range
lasso_paths = np.array([Lasso(alpha=a, max_iter=20000).fit(Xs_tr, y_tr).coef_ for a in alphas])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, paths, name in [(axes[0], ridge_paths, "Ridge: everything shrinks, nothing dies"),
                        (axes[1], lasso_paths, "Lasso: weak features hit exactly zero")]:
    for j, col in enumerate(X.columns):
        ax.plot(alphas, paths[:, j], label=col)
    ax.set_xscale("log"); ax.axhline(0, color="k", lw=.5); ax.set_xlabel("alpha"); ax.set_title(name)
axes[0].set_ylabel("standardised coefficient"); axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
# Which feature does Lasso switch off FIRST, and which survives LONGEST?
alive = (np.abs(lasso_paths) > 1e-8)
death_alpha = [alphas[np.argmin(alive[:, j])] if not alive[:, j].all() else np.inf for j in range(X.shape[1])]
print(pd.Series(death_alpha, index=X.columns).sort_values().rename("alpha at which Lasso zeroes it"))
print("\nThe last survivor is the feature the model trusts most. Does it match the standardised OLS coefficients from Section 2?")

## 7. Picking alpha honestly: cross-validation 🎯

Choosing alpha on the *test* set is cheating (you would be tuning on the exam paper). `RidgeCV` and `LassoCV` split the **training** data into folds, try each alpha, and pick the one with the best average validation score. Day 13 goes deep on this; today, just use it.

In [ ]:
models = {
    "OLS"      : make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge(1)" : make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso(.01)": make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=20000)),
    "RidgeCV"  : make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
    "LassoCV"  : make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=SEED, max_iter=20000)),
}
report = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    est = m[-1]
    chosen = getattr(est, "alpha_", getattr(est, "alpha", "-"))
    report.append({"model": name, "test_R2": m.score(X_te, y_te),
                   "test_RMSE": mean_squared_error(y_te, m.predict(X_te)) ** 0.5,
                   "zero_coefs": int((est.coef_ == 0).sum()), "alpha": chosen})
print(pd.DataFrame(report).set_index("model").round(4))
print("\nOn 16k rows × 8 features, regularization barely changes R². It shines when data is small or features are many/correlated.")

## 8. 🎯 Try it yourself: polynomial features on real data (10 min)

California prices are clearly **not** linear in `MedInc` (median income). Build a pipeline that:

1. keeps only the `MedInc` column,
2. adds polynomial features of degree **d** for d in `[1, 2, 3, 5]`,
3. standardises, then fits `Ridge(alpha=1.0)`,
4. prints train and test RMSE for each d.

Which degree would you ship? Why?

In [ ]:
# YOUR CODE HERE
X1_tr = X_tr[["MedInc"]]
X1_te = X_te[["MedInc"]]

for d in [1, 2, 3, 5]:
    # build the pipeline, fit on X1_tr / y_tr, print RMSE on train and test
    pass

<details><summary>Solution</summary>

```python
for d in [1, 2, 3, 5]:
    m = make_pipeline(PolynomialFeatures(degree=d), StandardScaler(), Ridge(alpha=1.0)).fit(X1_tr, y_tr)
    tr = mean_squared_error(y_tr, m.predict(X1_tr)) ** 0.5
    te = mean_squared_error(y_te, m.predict(X1_te)) ** 0.5
    print(f"degree {d}: train RMSE {tr:.4f} | test RMSE {te:.4f}")
```
Degree 2 or 3 gives a small but real gain over the line; degree 5 gains nothing more. Ship the simplest model that is within noise of the best one (degree 2).
</details>

## Homework starter 🏡

**Easy:** your own commute curve. **Medium:** Bengaluru house prices (Kaggle: `amitabhajoy/bengaluru-house-price-data`). **Stretch:** Kaggle *House Prices: Advanced Regression Techniques* submission with `LassoCV`.

The scaffold below covers the Medium task. Upload `Bengaluru_House_Data.csv` to Colab (folder icon on the left → upload) and run.

> 💼 **Career note:** a Kaggle public-leaderboard rank on House Prices is a concrete, verifiable line on a fresher resume. Recruiters click it.

In [ ]:
import os, re

CSV = "Bengaluru_House_Data.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    # --- cleaning: keep rows where total_sqft is a plain number (drop "2100 - 2850" ranges) ---
    df = df[df["total_sqft"].astype(str).str.fullmatch(r"\d+(\.\d+)?")].copy()
    df["total_sqft"] = df["total_sqft"].astype(float)
    df["bhk"] = df["size"].astype(str).str.extract(r"(\d+)").astype(float)   # "2 BHK" -> 2
    df = df.dropna(subset=["total_sqft", "bath", "bhk", "price"])
    df = df[(df["total_sqft"] / df["bhk"]) > 300]                                 # remove absurd listings
    feats = ["total_sqft", "bath", "bhk"]
    Xb, yb = df[feats], df["price"]                                               # price is in lakhs
    Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=SEED)

    print("VIF:\n", vif_by_hand(Xb_tr).round(2), "\n")
    for name, est in [("OLS", LinearRegression()), ("Ridge", RidgeCV(alphas=np.logspace(-3, 3, 25))),
                      ("Lasso", LassoCV(cv=5, random_state=SEED, max_iter=20000))]:
        m = make_pipeline(StandardScaler(), est).fit(Xb_tr, yb_tr)
        print(f"{name:6s} test RMSE = {mean_squared_error(yb_te, m.predict(Xb_te)) ** 0.5:.1f} lakh")
    # YOUR CODE HERE: try adding PolynomialFeatures(degree=2) before the scaler. Does RMSE drop?
else:
    print("Upload Bengaluru_House_Data.csv from Kaggle to run this cell (skipped for now).")

## Key takeaways ✅

* **Multiple regression** = one weight per feature, same loss, same `LinearRegression()`. Standardise before comparing weights.
* **Multicollinearity** makes coefficients unreliable, not predictions. Measure with **VIF = 1/(1 − R²ⱼ)**; > 5–10 means act.
* **Polynomial features** let a linear model bend. The degree is a complexity knob.
* **Underfitting** = high bias (both errors high). **Overfitting** = high variance (train low, test high). You can only see it on held-out data.
* **Ridge (L2)** shrinks, fixes multicollinearity. **Lasso (L1)** shrinks *and* selects (exact zeros). Choose **alpha by cross-validation**, never on the test set.
* Scale features inside a `Pipeline` so nothing leaks from test to train.

**Tomorrow (Day 06):** the same line goes through a sigmoid and starts answering yes/no questions with probabilities: **Logistic Regression**.